# nanochat.cpp session setup (Kaggle)

A manual session check and the `pi` agent.

This notebook loads the API keys from Kaggle secrets and installs the
distribution. The three pipeline notebooks (`pretrain.ipynb`, `eval.ipynb`, and
`posttrain.ipynb`) run their own setup, so this notebook is optional. Use it to
check a session, to warm the cache, or to run the `pi` agent.

Set the accelerator to T4 and turn Internet on.

## API keys

Kaggle secrets hold the credentials. Each secret name becomes an environment
variable. `HF_TOKEN` lets the data download run without a prompt. The other
keys authenticate the `pi` model provider. The notebook reports a
missing secret. The run continues.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

# Each Kaggle secret name becomes an environment variable of the same name.
API_KEY_NAMES = [
    "HF_TOKEN",
    "GEMINI_API_KEY",
    "OPENROUTER_API_KEY",
    "DEEPSEEK_API_KEY",
    "OPENCODE_API_KEY",
]

secrets = UserSecretsClient()
for name in API_KEY_NAMES:
    try:
        os.environ[name] = secrets.get_secret(name)
        print("set", name)
    except Exception:
        print("missing", name)

## Install

One cell installs the distribution and one cell configures the environment
(`docs/dependency-layering-plan.md` section 6.5). The install carries the shared
library, so nothing is cloned and nothing is built. This notebook's own modules
(`session`, `kaggle_setup`, `pi_agent`) live in the repository's `notebooks/`
directory beside this notebook.

In [ ]:
%pip install -q "nanochat-cpp[hub]"
# The install carries both import packages. D6 has not published the binary
# wheel of section 6.3 yet, so the shared library comes from the search order
# (docs/dependency-layering-plan.md section 6.3). When the wheel exists, pin
# the version here:
# %pip install -q "https://.../nanochat_cpp_cuda_sm75-0.1.0.whl"

In [ ]:
import os; os.environ.setdefault("NANOCHAT_SANDBOX_BACKEND", "none")

## Host

`session.setup` prefers the installed package: it resolves the shared library
for the T4 (`sm_75`) without a clone and without a build. When the package is
absent it falls back to the source tree, where `kaggle_setup.setup` runs
`tools/kaggle/bootstrap.sh`, reads the host contract, and builds the library;
the bootstrap also installs `pi` and Node.js.

In [ ]:
import subprocess

import session

sess = session.setup(arch="sm_75")
session.describe(sess)

pi_version = subprocess.run(["pi", "--version"], capture_output=True,
                            text=True).stdout.strip()
devices = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total",
     "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("pi:", pi_version)
print("devices:")
print(devices)

## Pi check

Run one small prompt through the agent. A provider error raises an exception
and prints the cause. An empty answer can no longer pass in silence.

The cell pins the provider and the bare model ID through the `PI_PROVIDER`
and `PI_MODEL` variables. Set the `PI_AGENT_PROVIDER` and `PI_AGENT_MODEL`
environment variables to change them. Tool activity updates one
cumulative line in place.

In [ ]:
import importlib

import pi_agent

# A pull does not reload an already imported module. Reload it, so a
# new commit takes effect without a kernel restart, then bind the name.
importlib.reload(pi_agent)
from pi_agent import run_pi

# Pin the provider and the bare model ID. Without them, Pi selects a
# provider automatically and can reject the model with
# "Model access is disabled".
PI_PROVIDER = os.environ.get("PI_AGENT_PROVIDER", "opencode-go")
PI_MODEL = os.environ.get("PI_AGENT_MODEL", "deepseek-v4.1-flash")

run_pi("Reply with the single word READY.", cwd=sess.repo,
       provider=PI_PROVIDER, model=PI_MODEL)